# Framework Comparison: RAGAS vs DeepEval
*Phase 3 of MedRAG-Eval*

Same RAG pipeline, same generated answers, same judge model (`qwen2.5:7b`) - 
evaluated by two different frameworks in a single session to ensure identical conditions.

*For pipeline details see 01_rag_pipeline.ipynb*  
*For standalone RAGAS evaluation see 02_ragas_evaluation.ipynb*

In [ ]:
!pip install matplotlib

In [ ]:
!pip install ragas==0.2.14 -q
!pip install langchain-google-vertexai -q
!pip install numpy==1.26.4 --only-binary=:all: -q

# Base packages for this section
!pip install -qU langchain-core langchain-text-splitters langchain-community langchain-chroma langchain-ollama beautifulsoup4 lxml

# If you hit: deepeval requires click<8.4.0 but click 8.4.1 is installed
# run the three commands below to fix and verify dependency conflicts.
!python -m pip install "click>=8.0.0,<8.4.0"
!python -m pip install --upgrade --force-reinstall "deepeval==4.0.4"
!python -m pip check

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama


llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:7b",
    temperature=0,
    max_tokens = 200
)

# Load data from Web
loader = WebBaseLoader(["https://www.webmd.com/cancer/what-is-chordoma",
                        "https://www.webmd.com/brain/spinal-muscular-atrophy",
                        "https://www.webmd.com/back-pain/spinal-stenosis",
                        "https://www.webmd.com/pain-management/pain-management-treatment-overview"])
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever(
    search_kwargs={"k": 10}
)

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}

    Be accurate and specific. If the answer is not in the context, say you don't know.

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()

## Data Collection
Answers and contexts generated once, shared between both frameworks.
This ensures a fair comparison - both evaluate identical inputs.

In [8]:
data_list = []

test_cases = [
    {"question": "which cells chordoma develops from?",
     "ground_truth": "notochord cells"},
    {"question": "which symtoms will patient have with chordoma compressing spinal cord in thoracic spine?",
     "ground_truth": "Numbness, tingling, or weakness in your legs, loss of bowel or bladder control"},
    {"question": "why doctors may order bone scan and lungs ct as part of workup of chordoma?",
     "ground_truth": "to check for potential spread of the tumor"},
    {"question": "which gene problem causes SMA?",
     "ground_truth": "SMN1"},
    {"question": "what happens with spinal cord without SMN protein?",
     "ground_truth": "spinal cord can't send strong signals to control   muscles. then  the muscle cells  slowly die"},
    {"question": "how SMA usually gets diagnosed in USA?",
     "ground_truth": "SMA is part of the recommended uniform screening Panel and is performed at birth"},
    {"question": "which gene therapy drugs are approved by fda in USA?",
     "ground_truth": "nusinersen (Spinraza), onasemnogene abeparvovec-xioi (Zolgensma), and risdiplam (Evrysdi)"},
    {"question": "which types of spinal stenosis there are?",
     "ground_truth": "cervical, lumbar, thoracic"},
    {"question": "what usually causes narrowing of spinal canal?",
     "ground_truth": "bone spurs and thickened ligaments"},
    {"question": "which symptom comes last, when the most severe spinal stenosis develops?",
     "ground_truth": "bowel and bladder control loss"},
    {"question": "what include laminectomy surgery?",
     "ground_truth": "removal of posterior part of the vertebral canal"},
    {"question": "what is the main downside of using NSAIDS?",
     "ground_truth": "risk of heart attack or stroke"},
    {"question": "For which type of pain doctors use  steroid injections to trigger points?",
     "ground_truth": "muscle pain and chronic pain involving tissue that surrounds muscle"},
    {"question": "what is the main advantage of intrathecal drug delivery?",
     "ground_truth": "less dose of pain medications needed and due to this patient will have less side effects"}

]

data = {
        "question": [],
        "answer": [],
        "contexts": [],
        "ground_truth": []
}

for tc in test_cases:
    docs = retriever.invoke(tc["question"])
    data["question"].append(tc["question"])
    data["contexts"].append([doc.page_content for doc in docs])
    data["answer"].append(chain.invoke(tc["question"]))
    data["ground_truth"].append(tc["ground_truth"])

data_list = [
    {
        "user_input": data["question"][i],
        "response": data["answer"][i],
        "retrieved_contexts": data["contexts"][i],
        "reference": data["ground_truth"][i]
    }
    for i in range(len(data["question"]))
]



## Judge Model Setup
Custom `OllamaModel` wrapper to connect local Qwen 2.5 7B to DeepEval.
Same model used as RAGAS evaluator - differences in results reflect framework behavior, not model differences.

In [9]:
from deepeval.models.base_model import DeepEvalBaseLLM
import ollama
import re

class OllamaModel(DeepEvalBaseLLM):
    def __init__(self, model_name="qwen2.5:7b"):
        self.model_name = model_name

    def load_model(self):
        return self

    def generate(self, prompt: str, schema=None):
        response = ollama.chat(
            model=self.model_name,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a JSON-only output machine. "
                        "Your response must be a single valid JSON object."
                        "Do not include ```json, ```, or any text before or after the JSON."
                        "Start your response with { and end with }."
                        "When asked for verdicts, always include both 'verdict' and 'reason' fields for each item."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            format="json",
            options={
                "temperature": 0,
                "num_ctx": 8192,
                "num_predict": 2048,
            },
        )
        content = response["message"]["content"]
        match = re.search(r'\{.*\}', content, re.DOTALL)
        if match:
            return match.group()
        return content

    async def a_generate(self, prompt: str, schema=None):
        return self.generate(prompt, schema)

    def get_model_name(self):
        return self.model_name

judge_model = OllamaModel(model_name="qwen2.5:7b")

## DeepEval Evaluation
4 metrics: Faithfulness, Answer Relevancy, Contextual Precision, Contextual Recall.

In [ ]:
import pandas as pd
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric, ContextualRecallMetric, AnswerRelevancyMetric, FaithfulnessMetric

results = []

metrics = [
    FaithfulnessMetric(threshold=0.7, model=judge_model, include_reason=True, async_mode=False),
    AnswerRelevancyMetric(threshold=0.7, model=judge_model, include_reason=True, async_mode=False),
    ContextualPrecisionMetric(threshold=0.7, model=judge_model, include_reason=True, async_mode=False),
    ContextualRecallMetric(threshold=0.7, model=judge_model, include_reason=True, async_mode=False),
]

for i in range(len(data["question"])):
    test_case = LLMTestCase(
        input=data["question"][i],
        actual_output=data["answer"][i],
        retrieval_context=data["contexts"][i],
        expected_output=data["ground_truth"][i]
    )
    
    row = {"question": data["question"][i][:60] + "..."}
    
    for metric in metrics:
        metric.measure(test_case)
        row[metric.__class__.__name__] = metric.score
    
    results.append(row)
    
#     print(f"QUESTION: {data['question'][i]}")
#     print(f"ANSWER: {data['answer'][i]}")
#     print(f"REFERENCE: {data['ground_truth'][i]}")
#     print("=" * 80)

# pd.DataFrame(results)

In [11]:
import sys
import types

vertexai_chat_module = types.ModuleType("langchain_community.chat_models.vertexai")
vertexai_chat_module.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = vertexai_chat_module

vertexai_llm_module = types.ModuleType("langchain_community.llms.vertexai")
vertexai_llm_module.VertexAI = type("VertexAI", (), {})
sys.modules["langchain_community.llms.vertexai"] = vertexai_llm_module

## RAGAS Evaluation
Same 4 metrics evaluated on identical data.
`ragas==0.2.14` pinned for local Ollama compatibility.

In [ ]:
from ragas.metrics import ContextPrecision, ContextRecall, AnswerRelevancy, Faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas import EvaluationDataset, evaluate
from ragas.run_config import RunConfig
from ragas.embeddings import LangchainEmbeddingsWrapper

import nest_asyncio
nest_asyncio.apply()

config = RunConfig(
    timeout=9000.0,       # Maximum time (in seconds) to wait for a single operation (Default is 180)
    max_workers=1
)


evaluator_embeddings = LangchainEmbeddingsWrapper(embedding)

evaluator_llm = LangchainLLMWrapper(llm)

evaluation_dataset = EvaluationDataset.from_list(data_list)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        Faithfulness(llm=evaluator_llm)
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=config
)
df = result.to_pandas()

## Side-by-Side Comparison

In [16]:


for i, row in df.iterrows():
    print(f"{i}. QUESTION: {row['user_input']}")
    print(f"ANSWER: {row['response']}")
    print(f"REFERENCE: {row['reference']}")
    print("=" * 80)

ragas_scores = df[['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']]

deepeval_scores = pd.DataFrame(results)[['FaithfulnessMetric', 'AnswerRelevancyMetric', 'ContextualPrecisionMetric', 'ContextualRecallMetric']]
deepeval_scores.columns = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']

diff = ragas_scores.values - deepeval_scores.values
diff_df = pd.DataFrame(diff, columns=['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall'])
diff_df.index = ragas_scores.index

print("=== RAGAS Results ===")
display(ragas_scores)

print("=== DeepEval Results ===")
display(deepeval_scores)

print("=== Difference (RAGAS - DeepEval) ===")
display(diff_df.style.background_gradient(cmap='RdYlGn', vmin=-1, vmax=1))

0. QUESTION: which cells chordoma develops from?
ANSWER: Chordoma develops from notochord cells that are left behind in the spine and skull after birth. These cells can undergo changes in their genes, leading to rapid division and the formation of chordoma tumors.
REFERENCE: notochord cells
1. QUESTION: which symtoms will patient have with chordoma compressing spinal cord in thoracic spine?
ANSWER: The context provided does not specify symptoms specifically for chordoma compressing the spinal cord in the thoracic region. However, it does mention that chordomas can press on nerves in the spine or brain, causing pain, numbness, or weakness. The symptoms depend on where the cancer is and how big it is.

Given this information, patients with chordoma compressing the spinal cord in the thoracic spine might experience:

- Pain
- Numbness
- Weakness

These symptoms could be localized to the area of compression or radiate depending on the extent of nerve involvement. For a precise list of symp

,faithfulness,answer_relevancy,context_precision,context_recall
0,1.000000,0.896372,1.000000,1.0
1,0.666667,0.000000,0.000000,0.5
2,1.000000,0.000000,0.000000,1.0
3,1.000000,0.841417,0.242063,1.0
4,1.000000,0.815645,0.891723,0.5
5,1.000000,0.641450,1.000000,1.0
6,1.000000,0.714387,1.000000,1.0
7,0.750000,0.929146,1.000000,1.0
8,1.000000,0.954272,1.000000,1.0
9,0.800000,0.953464,0.000000,1.0


=== DeepEval Results ===


,faithfulness,answer_relevancy,context_precision,context_recall
0,0.000000,0.600000,1.000000,1.0
1,1.000000,1.000000,1.000000,1.0
2,0.500000,0.000000,0.000000,1.0
3,1.000000,0.666667,0.555556,1.0
4,1.000000,1.000000,0.569444,0.5
5,0.500000,0.333333,0.528571,1.0
6,0.666667,1.000000,1.000000,1.0
7,1.000000,0.750000,1.000000,1.0
8,1.000000,0.800000,0.500000,1.0
9,0.500000,1.000000,0.645397,1.0


=== Difference (RAGAS - DeepEval) ===


,faithfulness,answer_relevancy,context_precision,context_recall
0,1.000000,0.296372,-0.000000,0.000000
1,-0.333333,-1.000000,-1.000000,-0.500000
2,0.500000,0.000000,0.000000,0.000000
3,0.000000,0.174751,-0.313492,0.000000
4,0.000000,-0.184355,0.322279,0.000000
5,0.500000,0.308117,0.471429,0.000000
6,0.333333,-0.285613,-0.000000,0.000000
7,-0.250000,0.179146,-0.000000,0.000000
8,0.000000,0.154272,0.500000,0.000000
9,0.300000,-0.046536,-0.645397,0.000000


## Key Findings

**Agreement between frameworks:**
- `context_recall` = 1.0 across all questions in both frameworks - the only metric where RAGAS and DeepEval fully agree. Retriever consistently finds relevant content regardless of how it is evaluated.

**Systematic divergence:**
- `context_precision`: RAGAS is generally stricter - penalizes irrelevant chunks ranked above relevant ones more aggressively than DeepEval.
- `faithfulness`: No consistent pattern - sometimes RAGAS is stricter (Q0: +1.0 difference), sometimes DeepEval is stricter (Q1: -0.33). Reflects different prompting strategies for the same concept.
- `answer_relevancy`: Largest single divergence on Q1 (-1.0) - same answer, opposite verdicts.

**Notable case - Question 1 (chordoma thoracic symptoms):**
Largest divergence across all metrics. This question requires clinical specificity (thoracic level → legs only, not arms) that is absent from the WebMD source. Ground truth was based on clinical expert knowledge, not source content. This created a mismatch that both frameworks handled differently - and neither correctly captured the full clinical picture. This question will be revisited in Phase 4 adversarial testing.

**Notable case - Question 2 (bone scan workup):**
Answer exists in source material but retriever failed to surface the relevant chunk (`context_precision` = 0.0 in both frameworks). In one evaluation run the model hallucinated a correct answer from background knowledge; in another it correctly said "I don't know." Both frameworks scored these differently despite identical conditions - demonstrating that non-determinism at temperature=0 affects evaluation reproducibility.

**Conclusion:**
Framework choice matters. RAGAS and DeepEval measure the same concepts differently - RAGAS tends to be stricter on retrieval quality, DeepEval rewards contextual honesty. For medical RAG systems, neither framework alone is sufficient. Human expert validation remains essential, particularly for clinically specific questions that exceed source granularity.

**Next steps:** see `04_adversarial.ipynb` for safety and red teaming evaluation.